# Milestone - Unit 4: the collaborative path -- readers who liked the books you liked also read these

## 1. Load the catalog, the interactions, the keyword documents, and the cold readers

The same generated substrate Units 1-3 ran on. We load the keyword bags only so the Unit-3
`LexicalRetrievalPath` (our content baseline) can be scored side by side with the new collaborative
path. `bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been
generated yet.

In [ ]:
import json
from collections import defaultdict

import bookrec
import pandas as pd

K = 10
RANDOM_SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

train_positive = [r for r in rows if r["split"] == "train" and r["label"] == 1]
{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "train_positives": len(train_positive),
    "train_label0_exposed_not_liked": sum(
        1 for r in rows if r["split"] == "train" and r["label"] == 0
    ),
    "cold_readers": len(cold_readers),
}

## 2. Build, fit, and register the item-item collaborative path alongside the others

`ItemItemRetrievalPath.fit(interactions, catalog=...)` keeps the exact `RetrievalPath` signature.
It reads the interaction log directly -- but only the **train positives** (`split == "train"` and
`label == 1`): a read/like is a positive, and there are no negative *ratings*. The `label == 0`
**train** rows are **not** leakage -- they are valid, exposure-sampled negatives (the reader was
shown the book and did not engage it) that live in the train split; the positives-only co-occurrence
simply *chooses to ignore* them, because a non-engagement carries no co-*liking* signal (Unit 5 will
put those negatives to work). What *is* leakage is the val/test rows -- the sealed holdout we grade
against -- and those never enter the similarity. From the train positives the path builds an
item x item **cosine co-occurrence** similarity: two books are similar when the same readers engaged
both. The registry now holds four signals -- chance, crowd counts, content words, and who-read-what.

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=RANDOM_SEED)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords, k1=1.2, b=0.75).fit(rows, catalog=catalog_ids)
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
registry.register(random_floor)
registry.register(popularity)
registry.register(lexical)
registry.register(cf)

fitted = cf.artifact()
similarity = fitted["similarity"]
item_ids = fitted["item_ids"]
items_with_neighbors = sum(1 for row in similarity if row.any())
{
    "registered": [path.artifact_name() for path in registry],
    "similarity_shape": list(similarity.shape),
    "n_neighbors_cap": fitted["n_neighbors"],
    "items_with_at_least_one_neighbor": items_with_neighbors,
    "items_with_no_train_positives": len(item_ids) - items_with_neighbors,
}

## 3. Score all four paths on the frozen `val` scoreboard

Same scoreboard as Units 1-3: positive `val` rows, `k = 10`, cold readers excluded, a reader's
positive `train` items form the seen set and are never recommended back. The collaborative path
scores each candidate by the summed similarity to the books the reader has already read.

In [ ]:
boards = {
    path.name: bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    for path in registry
}

floor_hit = boards["random"].hit_rate_at_k
popularity_hit = boards["popularity"].hit_rate_at_k
lexical_hit = boards["lexical"].hit_rate_at_k
cf_hit = boards["item-item"].hit_rate_at_k

{
    "scored_readers": boards["item-item"].readers,
    "random_floor_hit_at_10": round(floor_hit, 4),
    "popularity_hit_at_10": round(popularity_hit, 4),
    "lexical_hit_at_10": round(lexical_hit, 4),
    "item_item_cf_hit_at_10": round(cf_hit, 4),
    "cf_over_floor": round(cf_hit / floor_hit, 1),
    "cf_over_popularity": round(cf_hit / popularity_hit, 2),
    "cf_over_lexical": round(cf_hit / lexical_hit, 2),
}

Item-item CF lands around 0.252 hit@10: roughly 21x the random floor (~0.012), about **2.33x the
popularity path** (~0.108), and about **1.59x the lexical content path** (~0.158). Read that
honestly. This is the first **collaborative** path -- it reads who-read-what, not the catalog text --
and the first path to beat **both** popularity and content, by the widest margin so far. Unit 3's
lexical path already modestly edged popularity (~1.46x); collaborative filtering is the first to
beat both decisively.

It is not magic. The signal is complementary to content, not a replacement: CF can only reach a book
that at least one training reader engaged. Here **818 of the 2000 catalog books have no train
positives at all** (an all-zero similarity row), so CF can never recommend them -- the coverage
ceiling that Unit 6 cold-start and blending will address. It also sits below the latent matrix
factorization of Unit 5, which generalizes this same co-occurrence signal for the **warm** books --
those that already have interactions -- by filling in plausible similarities a raw pairwise count
never saw. But matrix factorization does **not** rescue those 818 cold books: an id-only
factorization cannot invent factors for an item with zero train positives, so that coverage gap
still waits for Unit 6's blending and Unit 9's feature-based item cold start.

## 4. One reader's history -> top CF recommendations and the neighbors that drove them

Pick a reader deterministically (the lowest-id warm reader) and ask the collaborative path for its
top 10. Each recommendation's score is the summed similarity to the books the reader has read, so we
can name the **co-occurring neighbor books** -- the ones in the reader's history most similar to each
recommendation -- that drove it. That is the explanation collaborative filtering gives for free.

In [ ]:
seen_by = defaultdict(set)
for row in rows:
    if row["split"] == "train" and row["label"] == 1 and row["item_id"] in catalog:
        seen_by[row["reader_id"]].add(row["item_id"])

cold_set = set(cold_readers)
reader_id = min(r for r in seen_by if r not in cold_set)
seen = seen_by[reader_id]

row_of = {item_id: idx for idx, item_id in enumerate(item_ids)}
# retrieve already ranks candidates by summed similarity AND drops the seen books, so it hands
# back the top-K directly -- no second rank(exclude=seen) pass is needed.
recommendations = cf.retrieve(reader_id, {"seen": seen}, K)

explained = []
for candidate in recommendations:
    rec_row = row_of[candidate.item_id]
    # The seen books most similar to this recommendation -- the neighbors that drove its score.
    neighbors = sorted(
        ((float(similarity[rec_row, row_of[item_id]]), item_id) for item_id in seen),
        reverse=True,
    )
    drivers = [(item_id, round(score, 3)) for score, item_id in neighbors[:3] if score > 0]
    book = catalog[candidate.item_id]
    explained.append(
        {
            "item_id": candidate.item_id,
            "title": book.title,
            "genres": book.fields["genres"],
            "cf_score": round(candidate.score, 3),
            "top_neighbor_titles": " | ".join(
                catalog[item_id].title for item_id, _ in drivers
            ),
        }
    )

print("reader", reader_id, "has read", len(seen), "books:")
for item_id in sorted(seen):
    print("  ", item_id, catalog[item_id].title, "|", catalog[item_id].fields["genres"])
print("ranked ids:", [candidate.item_id for candidate in recommendations])
pd.DataFrame(explained)


Every recommendation is explained by a handful of books already in the reader's history: the
collaborative path found them because *other* readers who engaged those same books went on to engage
these. That is the signal Units 1-3 could not see -- popularity ignores the individual, and the
lexical path only matches words. Collaborative filtering is complementary to content, not a
replacement. Unit 5's matrix factorization generalizes this co-occurrence into a low-rank latent
taste space, filling plausible gaps between **warm** books that have interactions but that no single
reader has co-read yet. It still cannot reach the 818 **cold** books with no train positives at all;
closing that coverage gap is the job of Unit 6's blending and Unit 9's feature-based item cold
start.